# 🖥️ AI IT Helpdesk Agent — Version 3: Agent + Tool Calling Loop

---

## What changed from V1?

| Feature | V1 (Manual Routing) | V3 (Agent + Tool Loop) |
|---|---|---|
| **Nodes** | 5 nodes (classifier + 3 handlers + response) | 2 nodes (`agent` + `tools`) |
| **Routing** | Manual conditional edges with a Python function | LLM decides which tool to call |
| **Tools** | Plain Python functions | `@tool` decorated functions |
| **Flow** | Linear: classify → handle → respond → END | Loop: agent ↔ tools until done |
| **Flexibility** | Must update router code for every new tool | Just add a new `@tool` — LLM figures it out |

In V3 we let the **LLM decide** which tool to call (and whether to call one at all). LangGraph's `ToolNode` and `tools_condition` give us a clean agent loop with minimal code.

> **Scope of this notebook:** Agent + Tool Calling Loop **only**.  
> No human-in-the-loop, no ticket creation, no memory/checkpointer — those come in V4.

---
## 1 · Install Dependencies

In [ ]:
%pip install -U langgraph langchain-openai python-dotenv

---
## 2 · Imports

In [ ]:
# LangGraph
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

# LangChain
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI

# Standard library / env
from dotenv import load_dotenv
import os

# For displaying the graph image
from IPython.display import Image, display

---
## 3 · Load Environment Variables

In [ ]:
# Load .env file (expects OPENAI_API_KEY at minimum)
load_dotenv()

---
## 4 · Create the LLM

In [ ]:
# Reads OPENAI_MODEL from .env, falls back to gpt-4.1-mini
model = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
    temperature=0
)

---
## 5 · Define Tools with `@tool`

Each tool returns the **same troubleshooting steps** we had in V1, but now they are proper LangChain tools.  
The **docstring** is critical — it tells the LLM *when* to use each tool.

In [ ]:
@tool
def password_help() -> str:
    """Get troubleshooting instructions for password reset issues.
    Use when user has forgotten password, is locked out, or needs to change password."""

    return (
        "Password Reset Steps:\n"
        "1. Go to https://passwordreset.company.com\n"
        "2. Click 'Forgot Password'\n"
        "3. Enter your employee ID\n"
        "4. Check your email for a reset link\n"
        "5. Create a new password (min 12 characters, 1 uppercase, 1 number, 1 special character)\n"
        "If you're still locked out after 30 minutes, contact IT Security at ext. 5555."
    )


@tool
def vpn_help() -> str:
    """Get troubleshooting instructions for VPN connection problems.
    Use when user cannot connect to VPN, has slow VPN, or VPN keeps disconnecting."""

    return (
        "VPN Troubleshooting Steps:\n"
        "1. Ensure you're connected to the internet first\n"
        "2. Restart the VPN client application\n"
        "3. Try connecting to a different VPN server (US-East, US-West, EU-Central)\n"
        "4. Check if your VPN client is up to date (current version: 5.2.1)\n"
        "5. Disable any other VPN or proxy software\n"
        "6. Restart your computer and try again\n"
        "If the issue persists, contact Network Support at ext. 4444."
    )


@tool
def general_it_help() -> str:
    """Get troubleshooting instructions for general IT issues.
    Use for slow computer, software crashes, printer problems, network issues,
    or any IT issue not related to VPN or passwords."""

    return (
        "General IT Troubleshooting Steps:\n"
        "1. Restart your computer (fixes 90%% of issues)\n"
        "2. Check if the issue occurs in Safe Mode\n"
        "3. Clear temporary files: Windows+R → type 'temp' → delete all files\n"
        "4. Run Windows Update to ensure your system is current\n"
        "5. Check available disk space (need at least 10GB free)\n"
        "6. Run a virus scan with the company antivirus software\n"
        "If the problem continues, submit a ticket at https://helpdesk.company.com\n"
        "or call the IT Help Desk at ext. 3333."
    )

---
## 6 · Bind Tools to the LLM

`bind_tools` tells the model about the available tools (their names, descriptions, and schemas).  
When the model thinks a tool should be called, it returns a **tool-call message** instead of plain text.

In [ ]:
# Collect all tools in a list
tools = [password_help, vpn_help, general_it_help]

# Bind them to the model
model_with_tools = model.bind_tools(tools)

---
## 7 · Understanding `MessagesState`

LangGraph provides a built-in state schema called **`MessagesState`**.  
Under the hood it is equivalent to:

```python
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]
```

The `add_messages` reducer **appends** new messages to the list (instead of replacing it).  
This gives us a growing conversation history automatically — no manual list management needed.

We use the built-in `MessagesState` directly so we don't have to write this ourselves.

---
## 8 · Define the Agent Node

The agent node prepends a **system prompt** and invokes the model (with tools bound).  
It returns the model's response — which may be either:
- A **tool-call message** (the model wants to call a tool), or
- A **regular AI message** (the model is done and has a final answer)

In [ ]:
# System prompt that guides the agent's behaviour
SYSTEM_PROMPT = """You are an IT Helpdesk Assistant. Your job is to help employees with IT support requests.

You have access to tools for:
- Password reset issues
- VPN connection problems
- General IT troubleshooting

Always use the appropriate tool first to get troubleshooting instructions.
Then give a short, clear, professional answer to the user.
Do not invent additional technical details."""


def agent_node(state: MessagesState):
    """Invoke the LLM with the system prompt + conversation history."""

    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]

    response = model_with_tools.invoke(messages)

    return {"messages": [response]}

---
## 9 · Build the Graph

The full graph has only **two nodes**:

| Node | What it does |
|---|---|
| `agent` | Calls the LLM; returns either a tool-call or a final answer |
| `tools` | Executes the requested tool(s) and returns the result |

`tools_condition` is a built-in LangGraph function that checks:
- If the last message has **tool calls** → route to `tools`
- Otherwise → route to `END`

In [ ]:
# Create the graph builder
builder = StateGraph(MessagesState)

# Add nodes
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))

# Add edges
builder.add_edge(START, "agent")                     # Entry point
builder.add_conditional_edges("agent", tools_condition)  # LLM decides: tools or END
builder.add_edge("tools", "agent")                   # After tool execution → back to agent

# Compile — NO checkpointer (memory comes in V4)
helpdesk_graph = builder.compile()

---
## 10 · Visualize the Graph

In [ ]:
display(Image(helpdesk_graph.get_graph().draw_mermaid_png()))

### How the Agent Loop works

```
START → agent → tools_condition?
                  ├── has tool calls → tools → agent  (loop back)
                  └── no tool calls  → END
```

1. The user's message enters the `agent` node.
2. The LLM sees the message + system prompt and decides whether to call a tool.
3. If it **does** call a tool → the `tools` node executes it → result goes back to `agent`.
4. The agent sees the tool result, formulates a human-friendly answer, and returns it.
5. `tools_condition` sees **no more tool calls** → routes to `END`.

This loop can repeat multiple times if the LLM calls several tools in sequence.

---
## 11 · Test: VPN Issue 🔌

In [ ]:
# Test with a VPN problem
result = helpdesk_graph.invoke(
    {"messages": [{"role": "user", "content": "My VPN is not connecting"}]}
)

result["messages"][-1].pretty_print()

---
## 12 · Test: Password Issue 🔑

In [ ]:
# Test with a password problem
result2 = helpdesk_graph.invoke(
    {"messages": [{"role": "user", "content": "I forgot my password"}]}
)

result2["messages"][-1].pretty_print()

---
## 13 · Test: General IT Issue 💻

In [ ]:
# Test with a general IT problem
result3 = helpdesk_graph.invoke(
    {"messages": [{"role": "user", "content": "My laptop is very slow"}]}
)

result3["messages"][-1].pretty_print()

---
## 14 · Inspect the Message Flow 🔍

Let's look at **every message** in the VPN result to understand the agent loop in action.  
You should see this sequence:

1. `HumanMessage` — the user's input
2. `AIMessage` — the LLM's tool-call request (content is empty; it has `tool_calls`)
3. `ToolMessage` — the tool's output (troubleshooting steps)
4. `AIMessage` — the LLM's final human-friendly answer

In [ ]:
for i, msg in enumerate(result["messages"]):
    print(f"[{i}] {msg.__class__.__name__}")
    print(f"    {msg.content[:100] if msg.content else '(tool call)'}")

    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"    Tool calls: {[tc['name'] for tc in msg.tool_calls]}")

    print()

---
## 15 · V1 vs V3 — Comparison

### Architecture Diagrams

**V1 — Manual Conditional Routing (5 nodes)**
```
START → classify_request → ┬─ "password"  → handle_password ─┐
                           ├─ "vpn"       → handle_vpn      ─┤
                           └─ "general"   → handle_general  ─┤
                                                              ↓
                                                       generate_response → END
```

**V3 — Agent + Tool Calling Loop (2 nodes)**
```
START → agent ←→ tools → agent → END
```

### Detailed Comparison

| Aspect | V1 | V3 |
|---|---|---|
| **Total nodes** | 5 | 2 |
| **Routing logic** | Hand-written `classify_request` function with keyword matching | LLM reads tool docstrings and decides |
| **Adding a new category** | Add a new handler node + update router + add edge | Just write a new `@tool` function |
| **Tool framework** | Plain Python functions | `@tool` decorator + `ToolNode` |
| **State** | Custom `TypedDict` with `issue_type`, `response`, etc. | Built-in `MessagesState` (just a message list) |
| **LLM calls** | 1 (response generation only) | 2+ (tool decision + final answer) |
| **Flexibility** | Low — rigid routing paths | High — LLM can chain multiple tools |
| **Error handling** | Must code a default branch | LLM gracefully handles unknown topics |

### Key Takeaway

> V1 tells the LLM **what to do**. V3 gives the LLM **tools** and lets it **decide**.

---
## 16 · What We Learned in V3

✅ **`@tool` decorator** — converts a function into a LangChain tool with schema auto-generated from the docstring  
✅ **`bind_tools`** — attaches tool definitions to the LLM so it knows what's available  
✅ **`MessagesState`** — built-in state that manages a growing message list with `add_messages`  
✅ **`ToolNode`** — prebuilt node that executes tool calls from the LLM's response  
✅ **`tools_condition`** — prebuilt router that checks for tool calls vs. final answer  
✅ **Agent loop** — the `agent ↔ tools` cycle that lets the LLM reason and act iteratively

---
## 17 · Next Steps → V4

V3 gives us a clean agent loop, but it's missing some important production features:

| Feature | V3 | V4 |
|---|---|---|
| **Human-in-the-Loop** | ❌ | ✅ Agent pauses for human approval before critical actions |
| **Ticket Creation** | ❌ | ✅ `create_ticket` tool for unresolved issues |
| **Memory / Checkpointer** | ❌ | ✅ Conversation persists across invocations |

**See you in Version 4!** 🚀